# Quand l'ACP trompe

| Module | Durée | Prérequis |
|---|---|---|
| M8 – Décomposition en valeurs singulières et données | 75 min | M6 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Un club d'athlétisme a mesuré ses 120 licenciés : taille (en cm), masse (en kg), temps au 100 m (en s), longueur au saut (en cm), et la discipline principale de chacun (sprint ou demi-fond). Un stagiaire en sciences du sport a lu que l'ACP « trouve automatiquement les informations importantes ». Il l'applique au tableau et présente fièrement ses conclusions à l'entraîneuse :

- « Un seul axe suffit : il porte 99 % de l'information. »
- « Cet axe, c'est la longueur du saut. Les autres mesures ne servent presque à rien. »

L'entraîneuse est sceptique. Les mesures sont simulées pour ce notebook, mais les pièges sont bien réels.

**Les questions : qu'est-ce qui a pu tromper le stagiaire ? Dans quels cas l'ACP donne-t-elle une image fausse ou incomplète des données, et comment s'en apercevoir ?**

**Activité** : Avant tout calcul, relevez ce qui vous semble suspect dans les deux affirmations du stagiaire.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_part_axe_1_apres_reduction = 0.99      # part du premier axe une fois les variables réduites (modifiez)

## ③ Abstraire : ce que l'ACP suppose

L'ACP cherche les directions de **plus grande dispersion**, mesurée par la somme des carrés. Cette définition cache plusieurs hypothèses :

1. **Les variables sont comparables.** La dispersion d'une variable dépend de son unité : la même longueur de saut, exprimée en centimètres, a une variance 10 000 fois plus grande qu'en mètres.
2. **Pas de valeurs aberrantes.** Les carrés donnent un poids énorme aux points très éloignés, comme pour les moindres carrés du module M6.
3. **La structure est « droite ».** Les axes sont des droites : une structure courbe ne se résume pas bien par quelques axes, même si elle est très simple.
4. **Ce qui varie beaucoup est ce qui compte.** Rien ne garantit que la direction utile pour **votre** question soit celle de plus grande dispersion.

![Schéma : trois pièges de l'ACP ; des unités différentes font suivre à l'axe 1 la variable aux plus grands nombres ; une structure courbe n'est captée par aucun axe droit ; deux groupes séparés selon la direction de plus faible dispersion](../../assets/schemas/schema-acp-pieges.svg)

**Le concept s'applique-t-il ?** Avant toute ACP, il faut se demander si ces hypothèses sont raisonnables pour les données et pour la question posée.

**Activité** : Une variable $x$ a pour écart type 0,6 m. Quel est son écart type si on l'exprime en centimètres ? Sa variance ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les données du club

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(84)
n = 120
sprint = rng.random(n) < 0.5
taille = rng.normal(176, 8, n)
masse = 0.9 * (taille - 100) + rng.normal(0, 5, n) + 3 * sprint
temps_100m = 12.6 - 0.9 * sprint - 0.02 * (taille - 176) + rng.normal(0, 0.35, n)
saut = 520 + 6 * (taille - 176) - 60 * (temps_100m - 12) + rng.normal(0, 25, n)
club = pd.DataFrame({"taille_cm": taille.round(1), "masse_kg": masse.round(1), "temps_100m_s": temps_100m.round(2),
                     "saut_cm": saut.round(0), "discipline": np.where(sprint, "sprint", "demi-fond")})
variables = ["taille_cm", "masse_kg", "temps_100m_s", "saut_cm"]
club[variables].describe().round(2)

### Piège 1 : des variables non réduites

In [ ]:
def acp(Y, reduire):
    X = Y - Y.mean(axis=0)
    if reduire:
        X = X / Y.std(axis=0)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return s ** 2 / np.sum(s ** 2), Vt, U * s

Y = club[variables].to_numpy()
for reduire in (False, True):
    part, Vt, _ = acp(Y, reduire)
    print(f"{'réduites' if reduire else 'brutes  '} : parts {', '.join(f'{p:.1%}' for p in part)}")
    print(f"           axe 1 = {', '.join(f'{c:+.2f} {v}' for c, v in zip(Vt[0], variables))}")

Y_m = Y.copy()
Y_m[:, 3] = Y_m[:, 3] / 100                     # le saut en mètres au lieu de centimètres
part_m, Vt_m, _ = acp(Y_m, reduire=False)
print(f"\nSaut exprimé en mètres, sans réduire : parts {', '.join(f'{p:.1%}' for p in part_m)}")
print(f"           axe 1 = {', '.join(f'{c:+.2f} {v}' for c, v in zip(Vt_m[0], variables))}")

**Activité** : Expliquez l'affirmation « 99 % de l'information » du stagiaire. Un résultat qui change quand on change d'unité peut-il décrire une propriété des athlètes ?

Indiquez vos réponses et justifiez-les.

### Piège 2 : une seule valeur aberrante

Une erreur de saisie s'est glissée : la taille de l'athlète 17 a été tapée 1 760 cm au lieu de 176 cm.

In [ ]:
Y_err = Y.copy()
Y_err[17, 0] = 1760                            # erreur de saisie : 1760 cm au lieu de 176
part_e, Vt_e, coord_e = acp(Y_err, reduire=True)
print(f"Avec l'erreur de saisie : parts {', '.join(f'{p:.1%}' for p in part_e)}")
print(f"axe 1 = {', '.join(f'{c:+.2f} {v}' for c, v in zip(Vt_e[0], variables))}")
print(f"Sans l'erreur             : parts {', '.join(f'{p:.1%}' for p in acp(Y, reduire=True)[0])}")
contribution = coord_e[17] ** 2 / np.sum(coord_e ** 2, axis=0)
for i in range(4):
    print(f"axe {i + 1} : part {part_e[i]:5.1%}, dont {contribution[i]:.0%} due au seul athlète 17")

### Piège 3 : une structure courbe

Le club suit un coureur sur la piste avec une montre GPS. Ses positions forment une boucle : une seule donnée, la distance parcourue sur le tour, suffit à les décrire.

In [ ]:
t = np.linspace(0, 2 * np.pi, 200, endpoint=False)
piste = np.column_stack([60 * np.cos(t), 35 * np.sin(t)]) + rng.normal(0, 0.8, (200, 2))
part_p, Vt_p, coord_p = acp(piste, reduire=False)
print(f"Parts des deux axes : {', '.join(f'{p:.0%}' for p in part_p)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(*piste.T, c=t, cmap="liouai", s=12); axes[0].set_aspect("equal"); axes[0].set_title("Positions GPS (couleur : moment du tour)")
axes[1].scatter(100 * t / (2 * np.pi), coord_p[:, 0], c=t, cmap="liouai", s=12)
axes[1].axhline(30, color=LIOUAI["jaune"], ls="--", lw=1)
axes[1].set_xlabel("avancement dans le tour (%)"); axes[1].set_ylabel("coordonnée sur l'axe 1 (m)")
axes[1].set_title("Résumer la position par l'axe 1 seulement")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Activité** : Sur le graphique de droite, la ligne pointillée coupe la courbe deux fois : que signifie-t-elle pour le résumé par l'axe 1 ? Que faudrait-il comme « coordonnée » pour décrire la boucle avec un seul nombre ?

Indiquez vos réponses et justifiez-les.

### Piège 4 : la direction utile n'est pas la plus dispersée

L'entraîneuse veut savoir si les mesures distinguent les sprinteurs des coureurs de demi-fond.

In [ ]:
part_r, Vt_r, coord_r = acp(Y, reduire=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (i, j) in zip(axes, [(0, 1), (0, 2)]):
    for disc, couleur in [("sprint", LIOUAI["vert"]), ("demi-fond", LIOUAI["navy"])]:
        m = (club["discipline"] == disc).to_numpy()
        ax.scatter(coord_r[m, i], coord_r[m, j], s=14, color=couleur, label=disc)
    ax.set_xlabel(f"axe {i + 1} ({part_r[i]:.0%})"); ax.set_ylabel(f"axe {j + 1} ({part_r[j]:.0%})"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for i in range(4):
    ecart = abs(coord_r[sprint, i].mean() - coord_r[~sprint, i].mean()) / coord_r[:, i].std()
    print(f"axe {i + 1} : part {part_r[i]:5.1%}   écart entre les deux disciplines : {ecart:.2f} écart type")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_part_axe_1_apres_reduction:.0%} ; part réelle du premier axe après réduction : {part_r[0]:.0%}")

**Activité** :

1. Rédigez pour le stagiaire une réponse bienveillante mais précise : quelles erreurs a-t-il commises, que fallait-il faire ?
2. Quel axe distingue le mieux les disciplines ? Est-ce le plus important au sens de l'ACP ? Que faut-il en conclure sur le mot « important » ?
3. Pour chacun des quatre pièges, proposez un contrôle simple à faire **avant** de croire une ACP.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **D'autres outils.** Pour séparer des groupes connus, il existe des méthodes conçues pour cela (l'analyse discriminante, par exemple) ; pour des structures courbes, des méthodes non linéaires. Pourquoi l'ACP reste-t-elle néanmoins un bon **premier** regard sur des données ?
- **Réduire, toujours ?** La réduction donne à chaque variable le même poids. Si une variable ne contient que du bruit de mesure, la réduction l'amplifie. Quand vaut-il mieux ne pas réduire ?
- **Généraliser.** Écrivez une fonction `diagnostic_acp(tableau)` qui, avant toute ACP, signale : les variables dont les écarts types diffèrent de plus d'un facteur 10 (réduction indispensable), les valeurs situées à plus de 5 écarts types de la moyenne (valeurs aberrantes possibles), et la part du premier axe après réduction. Testez-la sur le tableau du club, avec et sans l'erreur de saisie.

In [ ]:
def diagnostic_acp(tableau):
    # À compléter : renvoie une liste de messages
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def diagnostic_acp(tableau):
    Y = tableau.to_numpy(dtype=float)
    messages = []
    ecarts = Y.std(axis=0)
    if ecarts.max() / ecarts.min() > 10:
        messages.append(f"Écarts types de {ecarts.min():.2g} à {ecarts.max():.2g} : réduire les variables est indispensable.")
    z = (Y - Y.mean(axis=0)) / ecarts
    for i, j in zip(*np.nonzero(np.abs(z) > 5)):
        messages.append(f"Ligne {i}, variable {tableau.columns[j]} : {Y[i, j]:g} est à {z[i, j]:+.1f} écarts types de la moyenne.")
    part, _, _ = acp(Y, reduire=True)
    messages.append(f"Après réduction, le premier axe porte {part[0]:.0%} de la dispersion.")
    return messages

print("Tableau du club :")
for m in diagnostic_acp(club[variables]):
    print("  -", m)
club_err = club[variables].copy()
club_err.iloc[17, 0] = 1760
print("Avec l'erreur de saisie :")
for m in diagnostic_acp(club_err):
    print("  -", m)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Repérer les sophismes et mauvais usages du calcul": 1,
    "Lister les sources d'erreur liées aux limites des concepts": 1,
    "Reconnaître si le concept s'applique": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Sentir la fiabilité d'un modèle": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "8.4-quand-l-acp-trompe", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")